# Where Should I Live? – Transport mit GitHub Codespaces

**Alles laeuft in der Cloud.** Oeffne dieses Notebook **im Codespace**, nicht in `github.dev` und nicht im lokalen Python-Kernel.

Eure Dateien `transport_pipeline.py`, `transport_steps.ipynb`, `municapilites.csv` und `zvv_passes.csv` liegen direkt im GitHub-Repository (`main`) – **ohne** Unterordner. Die erzeugte Datenbank `transport.db` liegt ebenfalls dort, sobald du sie erzeugt hast.


## 0. Python-Pakete im Codespace installieren

Fuehre diese Zelle **einmal** mit `Shift + Enter` aus. Anschliessend kannst du die restlichen Zellen von oben nach unten starten.


In [ ]:
%pip install -q requests pandas ipykernel

## 1. Cloud-Umgebung pruefen und Python-Programm laden

In [ ]:
# Diese Zelle verwendet den Python-Kernel deines GitHub Codespace.
# Sie benoetigt keine Dateien auf deinem Mac und keinen GitHub-Token.
from pathlib import Path
import os
import sys

# Schutz vor dem vorherigen Fehler: github.dev ist kein Codespace.
if os.environ.get("CODESPACES") != "true":
    raise RuntimeError(
        "Falscher Python-Kernel. Oeffne GitHub > Code > Codespaces > "
        "Create codespace on main, und waehle dort den Python-Kernel. "
        "github.dev / ein lokaler Jupyter-Kernel kann die virtuellen GitHub-Dateien nicht lesen."
    )

# Wir suchen transport_pipeline.py im Hauptordner des Cloud-Repositories.
start = Path.cwd().resolve()
possible_folders = [start, *start.parents]
github_workspace = os.environ.get("GITHUB_WORKSPACE")
if github_workspace:
    possible_folders.append(Path(github_workspace))
repo = os.environ.get("GITHUB_REPOSITORY", "").split("/")[-1]
if repo:
    possible_folders.append(Path("/workspaces") / repo)

project_dir = next(
    (folder for folder in possible_folders if (folder / "transport_pipeline.py").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        "transport_pipeline.py wurde im Codespace nicht gefunden. "
        "Lege die Datei in den GitHub-Hauptordner neben transport_steps.ipynb; "
        f"aktuelles Cloud-Verzeichnis: {start}."
    )

# Python den Code-Ordner bekannt machen und unser Programm laden.
sys.path.insert(0, str(project_dir))
import transport_pipeline as tp

print("Transportprogramm geladen aus:", tp.BASE_DIR)
tp.show_cloud_environment()


## 2. CSV-Dateien pruefen

Das Programm erkennt auch die Schreibweisen `municipalities.csv` und `zvv_pass_prices.csv`.

In [ ]:
tp.check_input_files()

## 3. Winterthur → Zürich HB testen (Zug)

In [ ]:
tp.test_winterthur()

## 4. 20 Gemeinden anzeigen

In [ ]:
import pandas as pd  # Zeigt CSV-Daten als Tabelle an.
places = pd.read_csv(tp.municipality_file(), sep=None, engine="python")
display(places)


## 5. OeV-Fahrzeiten fuer alle Gemeinden in SQLite speichern

Das kann mehrere API-Abfragen ausloesen. Fehlende Verbindungen werden protokolliert.

In [ ]:
tp.build_pt_database()

## 6. SBB Open Data nach Tarifinformationen durchsuchen

Die Treffer sind **noch keine** offiziellen NetzPass-Preise.

In [ ]:
tp.research_sbb_prices()

## 7. ZVV-Abopreise 1. und 2. Klasse in die SQLite-Datenbank uebernehmen

Die vorhandene Preis-CSV muss beide Klassen und die geforderten Spalten enthalten.

In [ ]:
tp.update_pass_prices()

## 8. Fahrrad- und Autofahrzeiten via OpenRouteService (optional)

Verwende einen eigenen API-Key. Er wird verdeckt eingegeben und nicht in die GitHub-Dateien geschrieben.

In [ ]:
from getpass import getpass  # Key ohne sichtbaren Notebook-Text eingeben.
ors_key = getpass("OpenRouteService API-Key: ")
tp.add_bike_car_times(ors_key)


## 9. Ergebnisse ansehen – OeV-Klasse selber waehlen

Setze `fare_class` auf `1` fuer die 1. Klasse oder auf `2` fuer die 2. Klasse.

In [ ]:
fare_class = 2  # 1 = erste Klasse, 2 = zweite Klasse.
result = tp.show_database(fare_class=fare_class)
display(result)
print("Gemeinden:", len(result))


## 10. Wichtig: SQLite-Datei auf GitHub speichern

Die Datenbank `transport.db` wird zuerst im **Cloud-Dateisystem des Codespace** gespeichert. Das ist noch kein GitHub-Commit. Oeffne **Source Control** und waehle **Stage → Commit & Push** fuer die erzeugte Datei. Erst dann sehen die Teammitglieder und die Streamlit-App die neue Datenbank auf `main`.

Auch dein Codespace gehoert zu deinem Account. Er ist **nicht automatisch derselbe** wie der Codespace deiner Gruppenmitglieder.
